# Potential Talents - Candidate Ranking
## Notebook 01 - Data, duplicates, and rule-based relevance

This notebook addresses data preparation, transparent relevance scoring, and independent human validation.

In [1]:
import re, numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score
source=pd.read_csv('potential-talents.csv')
labels=pd.read_csv('unique-job-titles-labelled.csv').sort_values('representative_id').reset_index(drop=True)
assert len(source)==104 and source['job_title'].nunique()==52 and len(labels)==52
DIRECT=re.compile(r'\bhuman\s+resources\b|\bhr\b|\bhris\b|\bchro\b',re.I)
ADJ=re.compile(r'\bpeople\s+development\b|\btalent\s+(?:acquisition|management)\b|\bstaffing\b|\brecruit(?:er|ing|ment)\b|\bbenefits\b|\bcompensation\b',re.I)
SOLICIT=re.compile(r'\b(?:is|are)\s+seeking\b.*\b(?:professionals?|candidates?|applicants?)\b',re.I)
def H(t):
    t=str(t)
    if SOLICIT.search(t): return 0.0
    if DIRECT.search(t): return 1.0
    if ADJ.search(t): return 0.5
    return 0.0
def I(t,q):
    t,q=str(t).lower(),str(q).lower(); target='aspiring' if 'aspiring' in q else 'seeking'; other='seeking' if target=='aspiring' else 'aspiring'
    return 1.0 if target in t else (0.5 if other in t else 0.0)
def rule_score(t,q): return H(t)*(0.70+0.30*I(t,q))
def rule_level(t):
    h=H(t)
    if h==0:return 0
    if h<1:return 1
    tl=str(t).lower(); return 3 if ('aspiring' in tl or 'seeking' in tl) else 2
freq=source['job_title'].value_counts(); human=labels['manual_relevance_grade'].to_numpy(int); rule=labels['job_title'].map(rule_level).to_numpy(int)
print({'source_rows':len(source),'unique_titles':source['job_title'].nunique(),'repeated_unique_titles':int((freq>1).sum()),'max_frequency':int(freq.max()),'exact_agreement':round(float(np.mean(human==rule)),3),'kappa':round(float(cohen_kappa_score(human,rule,weights='quadratic')),3),'spearman':round(float(spearmanr(human,rule).statistic),3)})

### Verified results

- 104 source rows -> 52 unique job titles.
- 14 unique titles repeat; maximum exact-title frequency = 7.
- Human vs rule: 75.0% exact agreement, quadratic weighted kappa = 0.888, Spearman = 0.871.
- Rule equation: **R = H(0.70 + 0.30I)**.